In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn

# 禁用 JIT 调试（正式运行时可注释掉）
# jax.config.update('jax_disable_jit', True)

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = hi ** K

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
sampler = nk.sampler.MetropolisSampler(hi_ext, rule=single_rule, n_chains=16, sweep_size=32)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FCI 基准能量 (Ha)
E0 = -1.01546825  |  激发能 = 0.0000 eV
E1 = -0.87542794  |  激发能 = 3.8107 eV
E2 = -0.42938376  |  激发能 = 15.9482 eV
E3 = -0.26922131  |  激发能 = 20.3064 eV


以上内容不要修改